In [5]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

def load(path):
    images = {}
    files = [f for f in os.listdir(path) if f.lower().endswith('.png')]

    for file in files:
        fpath = os.path.join(path, file)
        
        img = cv2.imread(fpath)
        images[file] = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # BGR para RGB
            
    return images

set5, set14 = r'.\dataset\Set5', r'.\dataset\Set14'
set5, set14 = load(set5), load(set14)


In [6]:
def part(img):
    r = img[:, :, 0]
    g = img[:, :, 1]
    b = img[:, :, 2]
    channels = (r, g, b)
        
    return channels

In [7]:
def get_coord(h, w, scale_x, scale_y):

    # Calcula novas dimensões baseadas nas escalas
    h = int(h * scale_y)
    w = int(w * scale_x)
    
    # Cria a grade de coordenadas da nova imagem
    y, x = np.mgrid[0:h, 0:w]
    
    # Mapeia de volta para a imagem original
    y = (y + 0.5) / scale_y - 0.5
    x = (x + 0.5) / scale_x - 0.5
    
    return x, y

In [8]:
def nni(channel, scale_x, scale_y):
    h, w = channel.shape
    x, y = get_coord(h, w, scale_x, scale_y)
    
    y_near = np.round(y).astype(int)
    x_near = np.round(x).astype(int)
    
    y_near = np.clip(y_near, 0, h - 1)
    x_near = np.clip(x_near, 0, w - 1)
    
    return channel[y_near, x_near]

In [9]:
def bilinear(channel, scale_x, scale_y):
    h, w = channel.shape
    x, y = get_coord(h, w, scale_x, scale_y)
    
    x0 = np.floor(x).astype(int)
    y0 = np.floor(y).astype(int)

    x1 = np.clip(x0 + 1, 0, w - 1)
    y1 = np.clip(y0 + 1, 0, h - 1)
    
    x0 = np.clip(x0, 0, w - 1)
    y0 = np.clip(y0, 0, h - 1)
    
    dx = x - x0
    dy = y - y0
    
    v00 = channel[y0, x0]
    v01 = channel[y0, x1]
    v10 = channel[y1, x0]
    v11 = channel[y1, x1]
    
    v_top = v00 * (1 - dx) + v01 * dx
    v_bottom = v10 * (1 - dx) + v11 * dx
    
    return v_top * (1 - dy) + v_bottom * dy